# 02 — KL Divergence

Measure the cost of a wrong model: the *extra bits* you pay for coding with $q$ when the truth is $p$. You will build the formula from the mismatched code of `02/01`, prove by hand that this cost is never negative (Gibbs' inequality), and collect what that one fact buys: the uniform law is the most uncertain one, and no prefix code beats the entropy. Then you will meet the three ways relative entropy is *not* a distance: it is asymmetric, infinite off the support, and without a triangle inequality. What it obeys in place of the distance axioms is the subject of `02/03`.

**Prerequisites:** `02/01_shannon_entropy` (surprise, $H(p)$, prefix codes, Kraft's inequality, the source-coding floor and the cost of a mismatched code, today made exact), `01/15_fidelity_and_trace_distance` (total variation, a true distance; today's object is *not* one).

**What you'll be able to do**
- Define the relative entropy $D(p\|q) = \sum_x p_x\log_2\frac{p_x}{q_x}$ as *cross-entropy minus entropy*, compute it by hand, and read it as the extra bits per symbol of the wrong code: the $0.25$ bits of `02/01`'s mismatched codes, exactly.
- Prove Gibbs' inequality $D(p\|q) \ge 0$, with equality if and only if $p = q$, from the single inequality $\ln x \le x - 1$, and use it to re-prove $H(p) \le \log_2 n$ and, with Kraft's inequality from `02/01`, the "$\ge$" of the source-coding theorem.
- Show that $D$ is not a distance (asymmetric, $+\infty$ on a missing outcome, no triangle inequality), see each failure in a landscape over the triangle, and say why the asymmetry is a choice you will make in `03/11` and `04/08`.
- Read the bottom of the landscape: near its zero, relative entropy is half the quadratic form $\sum_i dp_i^2/p_i$ that `02/07` will name the Fisher metric, divided by $\ln 2$ when you count in bits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import kl_divergence, shannon_entropy

rng = np.random.default_rng(0)   # random distributions and random codes for the checks of section 2
viz.use_course_style()

## 1. Extra bits: relative entropy from the mismatched code

In `02/01` §4 you built a prefix code for the dyadic source $p = (\tfrac12, \tfrac14, \tfrac18, \tfrac18)$, with codeword lengths $\ell_x = -\log_2 p_x$ and average length $H(p) = 1.75$ bits, the floor. Then you used it on the *wrong* source: the uniform $u = (\tfrac14,\tfrac14,\tfrac14,\tfrac14)$ cost $2.25$ bits per symbol against $H(u) = 2$, and a $2$-bit code (the optimal code for $u$) on the dyadic source cost $2$ bits against $H(p) = 1.75$. Let's make that overpayment a formula.

A code built for a model $q$ spends $-\log_2 q_x$ bits on outcome $x$ (Shannon's lengths, `02/01`). If the data actually follow $p$, the average spend is the **cross-entropy**

$$
H(p, q) = -\sum_x p_x \log_2 q_x ,
$$

and the least you could have spent is $H(p) = -\sum_x p_x\log_2 p_x$. The difference is the **relative entropy**, or Kullback–Leibler divergence (Kullback & Leibler 1951),

$$
D(p\|q) = H(p, q) - H(p) = \sum_x p_x \log_2\frac{p_x}{q_x} ,
$$

the *extra bits per symbol* of believing $q$ when the truth is $p$. Read it "$p$ relative to $q$": truth first, model second. Term by term it is the average, under the truth, of the **surprise gap** $\log_2\frac{p_x}{q_x} = (-\log_2 q_x) - (-\log_2 p_x)$: how much more surprised the model is than it should be. Two conventions complete the definition. An outcome that never happens costs nothing, $0\log\frac{0}{q} = 0$; and $p\log\frac{p}{0} = +\infty$ for $p > 0$, a case section 3 returns to.

Units: $\log_2$ gives bits, as in `02/01`, and the helper's `base` argument switches to nats ($\ln$), the unit of the entropic transport of `03/09`–`03/11`.

By hand, the two mismatches of `02/01`. The dyadic code on the uniform source costs $D(u\|p) = \sum_x \tfrac14\log_2\frac{1/4}{p_x} = \tfrac14\big(\log_2\tfrac12 + \log_2 1 + \log_2 2 + \log_2 2\big) = \tfrac14(-1 + 0 + 1 + 1) = \tfrac14$ bit, which is the $2.25 - 2$. The two-bit code on the dyadic source costs $D(p\|u) = \sum_x p_x\log_2(4p_x) = \tfrac12\cdot 1 + \tfrac14\cdot 0 + \tfrac18\cdot(-1) + \tfrac18\cdot(-1) = \tfrac14$ bit, which is the $2 - 1.75$. Both equal $\tfrac14$ here; that is a coincidence of this pair, and section 3 shows that the two directions differ in general.

Now a pair that is not dyadic, used by the rest of this notebook and by `02/03`: the truth $p = (0.7, 0.2, 0.1)$ and the model $q = (0.4, 0.4, 0.2)$. By hand, the surprise gaps are $\log_2 1.75$ (a little above $0.8$), $-1$ and $-1$, so $D(p\|q) = 0.7\log_2 1.75 - 0.2 - 0.1$, a little below $0.27$ bits. Let's compute it term by term, against the cross-entropy and the helper `kl_divergence`, and draw the two surprises side by side.

In [ ]:
def kl_by_hand(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    if np.any((p > 0) & (q == 0)):              # the model gives zero to something that happens
        return np.inf
    nz = p > 0                                  # 0 log(0/q) = 0
    return float((p[nz] * np.log2(p[nz] / q[nz])).sum()) + 0.0

def cross_entropy(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    nz = p > 0
    return float(-(p[nz] * np.log2(q[nz])).sum())

p_dyadic = np.array([0.5, 0.25, 0.125, 0.125])
u4 = np.full(4, 0.25)
print("02/01's mismatches, made exact")
print(f"  dyadic code on the uniform source: cost H(u,p) = {cross_entropy(u4, p_dyadic):.3f}  =  H(u) {shannon_entropy(u4):.3f}  +  D(u||p) {kl_by_hand(u4, p_dyadic):.3f}")
print(f"  2-bit code on the dyadic source:   cost H(p,u) = {cross_entropy(p_dyadic, u4):.3f}  =  H(p) {shannon_entropy(p_dyadic):.3f}  +  D(p||u) {kl_by_hand(p_dyadic, u4):.3f}")

p = np.array([0.7, 0.2, 0.1])
q = np.array([0.4, 0.4, 0.2])
gaps = np.log2(p / q)
print("\nA three-outcome pair, term by term")
print("  p                     :", p)
print("  q                     :", q)
print("  surprise gap log2(p/q):", np.round(gaps, 4), "  (positive where the model under-rates the outcome)")
print("  weighted p*log2(p/q)  :", np.round(p * gaps, 4), "  -> sum =", round(float((p * gaps).sum()), 4), "bits")
print(f"  cross-entropy H(p,q) = {cross_entropy(p, q):.4f},  H(p) = {shannon_entropy(p):.4f},  difference = {cross_entropy(p, q) - shannon_entropy(p):.4f}")
print(f"  by hand D(p||q) = {kl_by_hand(p, q):.4f}   helper kl_divergence = {kl_divergence(p, q):.4f}   D(p||p) = {kl_by_hand(p, p):.4f}")
print(f"  in nats: helper base=e gives {kl_divergence(p, q, base=np.e):.4f}  =  bits x ln 2 = {kl_by_hand(p, q) * np.log(2):.4f}   (the unit of 03/09-03/11)")

# the figure: the two surprises per outcome, then the weighted gaps and their sum
labels = ["outcome 1", "outcome 2", "outcome 3"]
xpos = np.arange(3); width = 0.38
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.bar(xpos - width / 2, -np.log2(p), width, color=viz.SOURCE_COLOR, label="truth's surprise  -log2 p_x")
ax.bar(xpos + width / 2, -np.log2(q), width, color=viz.TARGET_COLOR, label="model's surprise  -log2 q_x")
for i in range(3):
    ax.text(xpos[i] - width / 2, -np.log2(p[i]) + 0.05, f"{-np.log2(p[i]):.2f}", ha="center", color=COLORS["text"])
    ax.text(xpos[i] + width / 2, -np.log2(q[i]) + 0.05, f"{-np.log2(q[i]):.2f}", ha="center", color=COLORS["text"])
ax.set_xticks(xpos, [f"{lab}\np = {p[i]}, q = {q[i]}" for i, lab in enumerate(labels)])
ax.set_ylabel("bits"); ax.set_ylim(0, 4.0)
ax.set_title("Surprise under the truth and under the model", pad=10)
ax.legend(loc="upper left")
ax = axes[1]
terms = p * gaps
bar_colors = [COLORS["positive"] if t > 0 else COLORS["negative"] for t in terms]
ax.bar(xpos, terms, 0.6, color=bar_colors)
ax.bar([3.3], [terms.sum()], 0.6, color=COLORS["highlight"])
for xi, t in zip(list(xpos) + [3.3], list(terms) + [terms.sum()]):
    ax.text(xi, t + (0.02 if t > 0 else -0.05), f"{t:+.4f}", ha="center", color=COLORS["text"])
ax.axhline(0.0, color=COLORS["muted"], lw=1)
ax.set_xticks(list(xpos) + [3.3], ["outcome 1", "outcome 2", "outcome 3", "sum = D(p || q)"])
ax.set_ylabel("bits"); ax.set_ylim(-0.3, 0.7)
ax.set_title("Weighted surprise gaps p_x log2(p_x / q_x), and their sum", pad=10)
fig.suptitle("Relative entropy is the average surprise gap: the model's surprise minus the truth's, weighted by the truth", y=1.02)
plt.show()

**Read the figure.** On the left, each outcome carries the surprise the truth assigns it (purple) and the surprise the model assigns it (amber). On outcome $1$ the model's bar is the taller one, $1.32$ bits against $0.51$: the model rates as uncommon an outcome that happens $70\%$ of the time. On outcomes $2$ and $3$ the model *over*-rates and is surprised one bit less than the truth. On the right, each gap is weighted by how often the truth produces the outcome: the positive term $+0.5651$ outweighs the negative $-0.2$ and $-0.1$, and the rose bar is their sum, $0.2651$ bits. The model can be less surprised than the truth on some outcomes, but on average, under the truth, it always pays. The next section proves it.

**Read the output.** The two overpayments of `02/01` are exactly $\tfrac14$ bit each, and each is a cross-entropy minus an entropy: $2.25 = 2 + 0.25$ and $2 = 1.75 + 0.25$. On the three-outcome pair the gaps are $+0.8074$, $-1$ and $-1$ bits, and the by-hand value, the helper and the cross-entropy minus the entropy all agree on $0.2651$ bits; $D(p\|p) = 0$, since the right model wastes nothing. In nats the same divergence is $0.1838$, the value in bits times $\ln 2$.

## 2. Gibbs' inequality, and what one inequality buys

**Claim.** $D(p\|q) \ge 0$, with equality if and only if $p = q$.

**Proof by hand.** Everything rests on one picture: $\ln t \le t - 1$ for all $t > 0$, with equality only at $t = 1$ (the tangent line at $1$ sits above the concave $\ln$). In one line: $g(t) = t - 1 - \ln t$ has $g'(t) = 1 - 1/t$, negative before $1$ and positive after, so its only minimum is $g(1) = 0$. Apply it to $t = q_x/p_x$ on the support of $p$:

$$
-D(p\|q)\,\ln 2 \;=\; \sum_x p_x \ln\frac{q_x}{p_x} \;\le\; \sum_x p_x\Big(\frac{q_x}{p_x} - 1\Big) \;=\; \sum_{x:\,p_x>0} q_x - 1 \;\le\; 0 .
$$

The first "$\le$" is an equality only if $q_x/p_x = 1$ wherever $p_x > 0$; the second only if $q$ puts no mass outside the support of $p$. Together: equality if and only if $p = q$. That is the whole proof (Cover & Thomas, §2.6).

Two promises of `02/01` are now due.

*The uniform is the most uncertain, a second proof.* Take $q = u = (\tfrac1n, \dots, \tfrac1n)$: $D(p\|u) = \sum_x p_x\log_2(n\,p_x) = \log_2 n - H(p) \ge 0$, so $H(p) \le \log_2 n$, with equality if and only if $p = u$. Jensen's inequality was `02/01`'s route; Gibbs is the shorter one, and it says *what* the gap is: $\log_2 n - H(p)$ is the relative entropy from $p$ to the uniform.

*Kraft's inequality, recalled.* Every prefix code satisfies $\sum_x 2^{-\ell_x} \le 1$: on the binary tree of `02/01` §4, the codewords own disjoint sets of nodes at the deepest level.

*The "$\ge$" of the source-coding theorem.* Take any prefix code with lengths $\ell_x$ and Kraft sum $Z = \sum_x 2^{-\ell_x} \le 1$. Define the model the code implicitly believes, $q_x = 2^{-\ell_x}/Z$, a probability vector. Then $\ell_x = -\log_2 q_x - \log_2 Z$, and the average length is
$$
\bar\ell = \sum_x p_x\ell_x = H(p, q) - \log_2 Z = H(p) + D(p\|q) - \log_2 Z \;\ge\; H(p),
$$
since $D \ge 0$ by Gibbs and $-\log_2 Z \ge 0$ by Kraft. Equality needs $D = 0$ (lengths exactly the surprises) and $Z = 1$ (no slack), which is possible only when every $p_x$ is a power of $\tfrac12$, as for the dyadic source of `02/01`. Every code pays its relative entropy to the source (Cover & Thomas, §5.3). Let's draw the tangent-line picture, then check both consequences numerically on random distributions and random codes.

In [ ]:
# the one inequality: ln x <= x - 1
xs = np.linspace(0.05, 3.0, 300)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(xs, np.log(xs), color=viz.SOURCE_COLOR, lw=2.2, label="ln x")
ax.plot(xs, xs - 1, color=viz.TARGET_COLOR, lw=2.2, label="x - 1  (tangent at x = 1)")
ax.fill_between(xs, np.log(xs), xs - 1, color=COLORS["highlight"], alpha=0.25, label="the gap: x - 1 - ln x >= 0")
ax.axvline(1.0, color=COLORS["muted"], lw=1, ls="--"); ax.axhline(0.0, color=COLORS["muted"], lw=1, ls="--")
ax.scatter([1.0], [0.0], color=COLORS["text"], zorder=5)
ax.set_xlabel("x"); ax.set_ylabel("value")
ax.set_title("Gibbs' inequality rests on one picture: ln x touches x - 1 only at x = 1", pad=12)
ax.legend(loc="lower right")
plt.show()

# consequence 1: D(p || uniform) = log2 n - H(p) >= 0, equality only at the uniform
n = 5
u = np.full(n, 1.0 / n)
ps = rng.dirichlet(np.ones(n), size=2000)
gap = np.array([kl_by_hand(pi, u) for pi in ps])
print(f"2000 random 5-outcome p:  max |D(p||u) - (log2 5 - H(p))| = {np.max(np.abs(gap - (np.log2(n) - np.array([shannon_entropy(pi) for pi in ps])))):.2e}")
print(f"                          min D(p||u) = {gap.min():.4f} bits (>= 0), D(u||u) = {kl_by_hand(u, u):.4f}")

# consequence 2: every prefix code pays D to the source.  Random Kraft-feasible lengths -> implicit model q
p5 = rng.dirichlet(np.ones(n))
worst = np.inf
for _ in range(2000):
    lengths = rng.integers(1, 8, size=n)
    Z = (2.0 ** -lengths).sum()
    if Z > 1:                      # not a prefix code (Kraft fails): skip
        continue
    q_code = 2.0 ** -lengths / Z
    avg_len = (p5 * lengths).sum()
    identity = shannon_entropy(p5) + kl_by_hand(p5, q_code) - np.log2(Z)
    assert abs(avg_len - identity) < 1e-12
    worst = min(worst, avg_len - shannon_entropy(p5))
print(f"random prefix codes on a random source: average length - H(p) >= {worst:.4f} bits in every case (the code's D(p||q) - log2 Z)")

**Read the figure.** The logarithm lies below its tangent at $1$ everywhere and touches it only there. The rose-shaded gap between the two curves, $x - 1 - \ln x$, is never negative, and that is Gibbs' inequality: evaluate the gap at $x = q_x/p_x$, weight it by $p_x$, and sum, and you get $D(p\|q)\ln 2$ *minus* the mass $q$ puts outside the support of $p$, when there is any. Since the gaps are never negative, $D(p\|q)\ln 2$ is at least that mass, and so at least $0$. To the left of $1$, where $\ln x$ plunges towards $-\infty$, the gap grows fast: an outcome the model rates far below the truth is expensive, a first glimpse of the walls of section 3.

**Read the output.** Two thousand random five-outcome distributions all satisfy $D(p\|u) = \log_2 5 - H(p)$ to machine precision, none goes below zero, and the uniform itself gives exactly $0$. For every Kraft-feasible code tried on a random source, the average length is exactly $H(p) + D(p\|q) - \log_2 Z$ (asserted to $10^{-12}$ inside the loop), so the excess over the entropy is never negative; the smallest excess found is $0.2544$ bits for this draw (the value depends on the source drawn). A code can match the floor only by believing the truth, with no slack. One inequality, three theorems.

## 3. Three ways it is *not* a distance

The total variation of `01/15` is a distance: symmetric, zero only at equality, and it obeys the triangle inequality (each property can be checked term by term: $|p_x - r_x| \le |p_x - q_x| + |q_x - r_x|$). Relative entropy keeps only the middle property. That is not a defect, since it is what a *coding cost* looks like, but you will want to know the three failures well, because each one reappears in the quantum lift (`02/10`) and in the transport problems of modules 03 and 04.

**(i) Asymmetry.** $D(p\|q)$ is the cost of believing $q$ when $p$ is true; $D(q\|p)$ is the opposite mistake. Nothing forces them to be equal, and they rarely are. Fix $p$ and let $q$ range over the whole triangle of three-outcome distributions: the two landscapes $q \mapsto D(p\|q)$ and $q \mapsto D(q\|p)$ are different surfaces that share one zero, at $q = p$.

**(ii) Infinity off the support.** If $q_x = 0$ for an outcome with $p_x > 0$, then $D(p\|q) = +\infty$: the model has *no codeword* for an event that happens, and no finite number of bits can describe it. The reverse, $p_x = 0$ where $q_x > 0$, costs nothing: $0\log\frac{0}{q} = 0$, a codeword you never use. So $D(p\|q) < \infty$ requires $\mathrm{supp}\,p \subseteq \mathrm{supp}\,q$. In the landscape, the edges of the triangle where $q$ has a zero are walls of infinite height for $D(p\|q)$. For $D(q\|p)$, with $p$ strictly positive, the edges are finite, and the three corners cost $\log_2\frac1{p_x}$.

**(iii) No triangle inequality.** There are triples with $D(p\|r) > D(p\|q) + D(q\|r)$: going through a midpoint can be cheaper than going direct. Take $r = (0.1, 0.2, 0.7)$, which puts most of its mass where $p$ puts the least. The output below prices the direct route and the detour through $q$.

*By hand.* $D(q\|p) = 0.4\log_2\tfrac47 + 0.4 + 0.2 = 0.6 - 0.4\log_2 1.75$, a little above $0.27$ bits, not the $0.2651$ of the other direction. With $r = (0.1, 0.2, 0.7)$: $D(p\|r) = 0.7\log_2 7 + 0 - 0.1\log_2 7 = 0.6\log_2 7 \approx 1.68$, while $D(q\|r) = 0.8 + 0.4 + 0.2\log_2\tfrac27 \approx 0.84$, so the two legs through $q$ cost about $1.10$ bits together, less than the direct route. And $D(q_{\text{hole}}\|p) = \tfrac12\log_2\tfrac{25}{14} \approx 0.42$ for $q_{\text{hole}} = (0.5, 0.5, 0)$.

The asymmetry is why you will *choose* a direction. Sinkhorn's algorithm (`03/10`), read as a KL projection in `03/11`, minimises $D(\pi\|K)$, plan first and kernel second, and that choice decides what the solution looks like. Let's draw the two landscapes, walk into an edge in both directions, and exhibit all three failures in numbers. Two shorter checks close the section: near its zero the landscape is a quadratic bowl, $D(p\|p + hv) \approx \tfrac{1}{2\ln 2}\sum_i (hv_i)^2/p_i$, and relabelling the outcomes changes nothing.

In [ ]:
def kl_landscapes(p, m=300):
    # q = (q1, q2, 1 - q1 - q2) on a grid over the simplex; both directions of D
    g = np.linspace(0.0, 1.0, m)
    Q1, Q2 = np.meshgrid(g, g)
    Q3 = 1.0 - Q1 - Q2
    inside = Q3 >= 0
    fwd = np.full(Q1.shape, np.nan); bwd = np.full(Q1.shape, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.stack([Q1, Q2, Q3], axis=-1)
        fwd[inside] = np.sum(p * np.log2(p / Q[inside]), axis=-1)                 # D(p || q): +inf on the edges
        Qi = Q[inside]
        terms = np.where(Qi > 0, Qi * np.log2(Qi / p), 0.0)                       # 0 log 0 = 0
        bwd[inside] = terms.sum(axis=-1)                                          # D(q || p): finite, p > 0
    return Q1, Q2, fwd, bwd

Q1, Q2, fwd, bwd = kl_landscapes(p)
vmax = 2.0
fig, axes = plt.subplots(1, 3, figsize=(16, 5.0), gridspec_kw={"width_ratios": [1, 1, 0.95], "wspace": 0.35})
for ax, Z, title in [(axes[0], fwd, "D(p || q): model q varies"), (axes[1], bwd, "D(q || p): truth q varies")]:
    im = ax.contourf(Q1, Q2, np.clip(Z, 0, vmax), levels=np.linspace(0, vmax, 21), cmap=viz.CMAP_COST)
    ax.contour(Q1, Q2, np.clip(Z, 0, vmax), levels=[0.1, 0.25, 0.5, 1.0], colors=[COLORS["text"]], linewidths=0.7)
    ax.plot([0, 1, 0, 0], [0, 0, 1, 0], color=COLORS["muted"], lw=1)
    ax.scatter([p[0]], [p[1]], color=COLORS["highlight"], s=60, zorder=5, label="q = p  (the only zero)")
    ax.set_xlabel("q1"); ax.set_ylabel("q2"); ax.set_title(title, pad=10); ax.set_aspect("equal")
    ax.legend(loc="upper right")
fig.colorbar(im, ax=axes[:2], shrink=0.85, label="bits (clipped at 2)")
# third panel: walk from p straight into the q3 = 0 edge, both directions
q_hole = np.array([0.5, 0.5, 0.0])
q3_walk = np.logspace(-1, -8, 300)                    # third entry of q_t, from p's 0.1 down towards 0
walk = [(1 - t) * p + t * q_hole for t in 1 - q3_walk / p[2]]   # q_t = (1-t) p + t q_hole has q3 = (1-t) p3
ax = axes[2]
ax.plot(q3_walk, [kl_by_hand(p, qt) for qt in walk], color=viz.SOURCE_COLOR, lw=2.4, label="D(p || q_t): diverges like 0.1 log2(0.1/q3)")
ax.plot(q3_walk, [kl_by_hand(qt, p) for qt in walk], color=viz.TARGET_COLOR, lw=2.4, label="D(q_t || p): finite, -> 0.418")
ax.axhline(kl_by_hand(q_hole, p), color=viz.TARGET_COLOR, lw=1, ls="--")
ax.set_xscale("log"); ax.invert_xaxis()
ax.set_xlabel("third entry q3 of q_t  (log axis, towards the edge)"); ax.set_ylabel("bits")
ax.set_title("Walking into the edge q3 = 0", pad=10); ax.legend(loc="upper left")
for a_ in axes[:2]:
    a_.plot([p[0], q_hole[0]], [p[1], q_hole[1]], color=COLORS["text"], lw=1, ls=":")
fig.suptitle("Two landscapes, one shared zero: relative entropy is asymmetric, and infinite off the support", y=1.04)
plt.show()

print(f"(i)  asymmetry:  D(p||q) = {kl_by_hand(p, q):.4f}   D(q||p) = {kl_by_hand(q, p):.4f}   (p = {p}, q = {q})")
print(f"(ii) support:    D(p||q_hole) = {kl_by_hand(p, q_hole)}   helper: {kl_divergence(p, q_hole)}   (q_hole = {q_hole} has no codeword for outcome 3, which p uses 10% of the time)")
print(f"                 D(q_hole||p) = {kl_by_hand(q_hole, p):.4f}   (the unused codeword costs nothing)")
print(f"                 corner costs of the middle panel: log2(1/p) = {np.round(np.log2(1 / p), 4)} bits at the three vertices")
print(f"                 largest value of D(q||p) on the grid of the middle panel: {np.nanmax(bwd):.4f} bits (finite; the colour scale clips at {vmax})")
edge = [kl_by_hand(np.array([0.0, s, 1 - s]), p) for s in np.linspace(0, 1, 1001)]   # the q1 = 0 edge
print(f"                 along the q1 = 0 edge of the middle panel, D(q||p) runs from {min(edge):.4f} to {max(edge):.4f} bits")
r = np.array([0.1, 0.2, 0.7])
print(f"(iii) triangle:  D(p||r) = {kl_by_hand(p, r):.4f}  >  D(p||q) + D(q||r) = {kl_by_hand(p, q):.4f} + {kl_by_hand(q, r):.4f} = {kl_by_hand(p, q) + kl_by_hand(q, r):.4f}")

# the bottom of the bowl: in nats the Hessian at q = p is the Fisher matrix; in bits it is divided by ln 2
v = np.array([0.5, -0.2, -0.3])                       # a tangent direction: entries sum to 0
for h in [1e-2, 1e-3, 1e-4]:
    ds2 = float(np.sum((h * v) ** 2 / p))             # Fisher quadratic form of 02/07
    print(f"near the zero, h = {h:.0e}:  D(p||p + h v) in bits / (ds^2 / 2) = {kl_by_hand(p, p + h * v) / (0.5 * ds2):.4f}   in nats: {kl_divergence(p, p + h * v, base=np.e) / (0.5 * ds2):.4f}")
print(f"1 / ln 2 = {1 / np.log(2):.4f}")

# relabelling the outcomes changes nothing
perm = [2, 0, 1]
print(f"relabelled outcomes: D(p[perm] || q[perm]) = {kl_by_hand(p[perm], q[perm]):.4f}  =  D(p||q) = {kl_by_hand(p, q):.4f}")

**Read the figure.** The two landscapes share the fixed $p = (0.7, 0.2, 0.1)$ and the single zero at the rose point $q = p$, and differ everywhere else. On the left ($D(p\|q)$, the model varies) the wall stands plainly along the $q_1 = 0$ edge, where the model denies the outcome that $p$ uses $70\%$ of the time. Along the other two edges the wall is there too but thin, narrower than the grid, because $p_2$ and $p_3$ are small. So the third panel walks straight at the $q_3 = 0$ edge along the dotted segment from $p$ to $q_{\text{hole}} = (0.5, 0.5, 0)$, and there the two directions part company: $D(p\|q_t)$ climbs without bound, slowly, like $0.1\log_2(0.1/q_3)$ (hence the logarithmic axis), while $D(q_t\|p)$ ends at the finite $0.418$ bits of $q_{\text{hole}}$. In the middle ($D(q\|p)$, the truth varies) the edges are **finite** everywhere, because a truth that never uses an outcome wastes nothing on it, and the corners are exactly $\log_2\frac1{p_x}$. The $q_1 = 0$ edge looks as dark as a wall only because its values, between $1.74$ and $3.32$ bits, reach past the colour scale's clip at $2$.

**Read the output.** Asymmetry: $D(p\|q) = 0.2651$ bits but $D(q\|p) = 0.2771$. Support: $D(p\|q_{\text{hole}})$ is `inf` for `kl_by_hand` and the helper alike, while the reverse costs $0.4183$ bits; the largest value in the middle panel is the $3.32$ bits of the corner $(0, 0, 1)$. Triangle: the detour $p \to q \to r$ costs $0.2651 + 0.8385 = 1.1037$ bits, the direct $p \to r$ costs $1.6844$. A coding cost, not a ruler.

The bottom of the bowl is where relative entropy meets the geometry of the triangle. Near its zero, $D(p\|p + h v)$ is quadratic in the step, and the printed ratio says which quadratic. In nats it settles on $1$ times half the form $ds^2 = \sum_i dp_i^2/p_i$ of `02/07`: the matrix of second derivatives (the Hessian) of $q \mapsto D(p\|q)$ at $q = p$ is the Fisher metric $\mathrm{diag}(1/p_i)$ on directions whose entries sum to zero, when you count in nats. In bits every value is divided by $\ln 2$, so the ratio settles on $1/\ln 2 = 1.4427$, and the Hessian is the Fisher metric divided by $\ln 2$. That is why the two landscapes look alike near their shared zero: both are the same quadratic bowl there. Fisher–Rao, the geometry that *does* make the triangle a metric space, is `02/07`.

The last line shows that relabelling the outcomes leaves relative entropy unchanged: it compares probabilities outcome by outcome and is blind to any geometry of the outcomes themselves, which is why `02/09` will call it a *vertical* comparison.

## Your turn

1. **Both ways, and a detour.** Pick your own $p, q$ on three outcomes and confirm $D(p\|q) \ne D(q\|p)$. Then search a few hundred random triples for the triple $p, q, r$ where $D(p\|q) + D(q\|r) - D(p\|r)$ is most negative, the worst triangle-inequality failure you can find, and describe the three distributions: where does $q$ sit relative to $p$ and $r$?
2. **The hole, and how practitioners fill it.** With `p = [0.7, 0.2, 0.1]` and `q = [0.5, 0.5, 0.0]`, $D(p\|q) = \infty$. Smooth the model, $q_\epsilon = (1 - \epsilon)q + \epsilon u$, and plot $D(p\|q_\epsilon)$ against $\epsilon$ on a log axis from $10^{-6}$ to $1$. How fast does the cost fall as the model admits the missing outcome, and what is the price at $\epsilon = 1$, the model that admits everything equally? This is why estimated distributions are smoothed before a KL is taken, and why the Gibbs kernel $K = e^{-C/\varepsilon}$ of `03/09` is strictly positive by construction.
3. **The binary relative entropy.** For two coins, $d(p\|q) = p\log_2\frac pq + (1-p)\log_2\frac{1-p}{1-q}$. Fix $q = 0.3$ and plot $d(p\|q)$ and $d(q\|p)$ for $p \in (0, 1)$ on the same axes; mark where one of them blows up and why the other does not, and check $d(p\|\tfrac12) = 1 - h(p)$ from `02/01`. Then expand $d(q + \delta\,\|\,q)$ to second order in $\delta$ and show that it is $\frac{\delta^2}{2\ln 2\;q(1-q)}$: the Fisher information of a coin, $1/q(1-q)$, divided by $\ln 2$, exactly as section 3 found on three outcomes.
4. **Harder: Pinsker's inequality, and why no reverse bound exists.** `01/15` measured total variation $\mathrm{TV}(p, q) = \tfrac12\sum_x|p_x - q_x|$. The inequality known as Pinsker's reads $D(p\|q) \ge \frac{2}{\ln 2}\,\mathrm{TV}(p, q)^2$ in bits. (a) Check it on $20{,}000$ random pairs of four-outcome distributions and find the pairs that come closest to equality; what do they have in common? (b) Show that the constant $\frac{2}{\ln 2}$ cannot be raised, using two coins $p = \tfrac12 + \delta$ and $q = \tfrac12$ with $\delta \to 0$ and the expansion of exercise 3. (c) Show that the reverse bound is impossible by exhibiting a family with $\mathrm{TV}$ bounded (tending to $\tfrac12$) while $D \to \infty$ (hint: section 3(ii)). A small relative entropy forces a small total variation; the converse fails, since the total variation can stay bounded while the relative entropy grows without bound.

## What you built

- You can define the relative entropy as cross-entropy minus entropy, compute it term by term by hand, and read each term as a surprise gap weighted by the truth: `02/01`'s two overpayments are exactly $\tfrac14$ bit each, $D(u\|p)$ and $D(p\|u)$, and the three-outcome pair costs $0.2651$ bits.
- You can prove Gibbs' inequality from $\ln x \le x - 1$ and spend it twice: $H \le \log_2 n$, with the gap named as $D(p\|u)$, and the "$\ge$" of the source-coding theorem, where every prefix code pays $D(p\|q) - \log_2 Z$ over the floor and Kraft's $Z \le 1$ comes from the tree of `02/01`.
- You can show the three failures of a distance (asymmetry, $+\infty$ off the support, no triangle inequality) in two landscapes and three numbers, and say why a coding cost has them.
- You can tell a wall from a finite edge: $D(p\|q)$ is infinite where the model denies an outcome, while $D(q\|p)$ stays finite, at most $3.32$ bits here, at the corner where $p$ puts the least mass.
- You can read the bottom of the bowl: near its zero, relative entropy is half the Fisher form of `02/07` in nats, and that half-form divided by $\ln 2$ in bits.

One inequality, $\ln x \le x - 1$, carried this notebook. `02/03` shows the laws the same quantity obeys in place of the distance axioms.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| relative entropy (Kullback–Leibler) $D(p\|q) = \sum_x p_x\log_2\frac{p_x}{q_x}$ | Umegaki relative entropy $S(\rho\|\sigma) = \mathrm{tr}\,\rho(\log_2\rho - \log_2\sigma)$ (`02/10`) |

## What's next

Relative entropy is not a distance, so why does the course build on it? Because it obeys three laws a coding cost obeys in place of the distance axioms: it adds up over independent symbols, as codeword lengths do; it is jointly convex, which makes the KL projections of `03/11` and `04/08` well posed; and no processing can increase it. `02/03_laws_of_relative_entropy` proves all three by hand and sends relative entropy on to the mutual information of `02/04`, the Sinkhorn projection of `03/11` and the Umegaki lift of `02/10`.

## References

- S. Kullback & R. A. Leibler, *On information and sufficiency*, Ann. Math. Statist. **22**, 79–86 (1951) — the definition. doi:10.1214/aoms/1177729694
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006) — §2.3 "Relative Entropy and Mutual Information", §2.6 "Jensen's Inequality and Its Consequences", §5.2 "Kraft Inequality", §5.3 "Optimal Codes". doi:10.1002/047174882X
- C. E. Shannon, *A mathematical theory of communication*, Bell Syst. Tech. J. **27**, 379–423 (1948) — the coding theorem whose converse is today's section 2. doi:10.1002/j.1538-7305.1948.tb01338.x
- M. S. Pinsker, *Information and Information Stability of Random Variables and Processes*, Holden-Day (1964) — the inequality known by his name, first proved there with a weaker constant; the constant $\frac{2}{\ln 2}$ of exercise 4 (in bits) is the sharp one, proved later by other authors.
- H. Umegaki, *Conditional expectation in an operator algebra, IV (entropy and information)*, Kodai Math. Sem. Rep. **14**, 59–85 (1962) — the quantum relative entropy of `02/10`, the quantum column of today's dictionary row. doi:10.2996/kmj/1138844604

**Previous:** `notebooks/02_InformationAndGeometry/01_shannon_entropy.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/03_laws_of_relative_entropy.ipynb`